<h4><b>Redes Neurais Artificiais</b></h4>
<p>Cross-validation</p>

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, learning_curve, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt

In [3]:
path = '/media/marcos/500GB/00_datasets/post_processing/post_processing_dataset.csv'
df = pd.read_csv(path)

In [4]:
df = df[[
    'line_category',
    'line_color',
    'station_initial_latitude',
    'station_initial_longitude',
    'station_final_latitude',
    'station_final_longitude',
    'distance',
    'average_speed_kh',
    'duration'
]]

In [6]:
X = df.iloc[:,0:8].values
y = df.iloc[:,8].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

scaler_x = StandardScaler()
X_train_scaled = scaler_x.fit_transform(X_train)
X_test_scaled = scaler_x.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

regressor_rna = MLPRegressor(max_iter=500, hidden_layer_sizes=(4,4), random_state=0)

cv = KFold(n_splits=5, shuffle=True, random_state=0)

In [7]:
def mae_inversed(estimator, X, y):
    y_pred_scaled = estimator.predict(X)
    y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()
    y_true = scaler_y.inverse_transform(y.reshape(-1, 1)).ravel()
    return mean_absolute_error(y_true, y_pred)

mae_scores_train = cross_val_score(regressor_rna, X_train_scaled, y_train_scaled, cv=cv, scoring=mae_inversed)

In [8]:
regressor_rna.fit(X_train_scaled, y_train_scaled)

MLPRegressor(hidden_layer_sizes=(4, 4), max_iter=500, random_state=0)

In [9]:
y_train_pred_scaled = regressor_rna.predict(X_train_scaled)
y_test_pred_scaled = regressor_rna.predict(X_test_scaled)

In [10]:
y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled.reshape(-1, 1)).ravel()
y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).ravel()

In [35]:
import numpy as np

mse_scores_test = []
rmse_scores_test = []
mae_scores_test = []

for train_idx, val_idx in cv.split(X_test_scaled, y_test_scaled):
    X_val, y_val = X_test_scaled[val_idx], y_test_scaled[val_idx]

    y_val_pred_scaled = regressor_rna.predict(X_val)
    y_val_pred = scaler_y.inverse_transform(y_val_pred_scaled.reshape(-1, 1)).ravel()
    y_val_true = scaler_y.inverse_transform(y_val.reshape(-1, 1)).ravel()

    mae_scores_test.append(mean_absolute_error(y_val_true, y_val_pred))
    mse_scores_test.append(mean_squared_error(y_val_true, y_val_pred))
    rmse_scores_test.append(np.sqrt(mse_scores_test[-1]))

# Treinando o modelo final
regressor_rna.fit(X_train_scaled, y_train_scaled)

# Fazendo previsões
y_train_pred_scaled = regressor_rna.predict(X_train_scaled)
y_test_pred_scaled = regressor_rna.predict(X_test_scaled)

# Convertendo de volta para a escala original
y_train_pred = scaler_y.inverse_transform(y_train_pred_scaled.reshape(-1, 1)).ravel()
y_test_pred = scaler_y.inverse_transform(y_test_pred_scaled.reshape(-1, 1)).ravel()

# Calculando métricas finais
mae_train = mean_absolute_error(y_train, y_train_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)

mse_train = mean_squared_error(y_train, y_train_pred)
mse_test = mean_squared_error(y_test, y_test_pred)

rmse_train = np.sqrt(mse_train)
rmse_test = np.sqrt(mse_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

# Calculando desvios padrão corretamente
mae_train_std = np.std(mae_scores_train)
mse_train_std = np.std(mse_scores_train)
rmse_train_std = np.std(rmse_scores_train)

mae_test_std = np.std(mae_scores_test)
mse_test_std = np.std(mse_scores_test)
rmse_test_std = np.std(rmse_scores_test)

# Exibindo os resultados
print("MAE para cada fold na Cross-Validation (dados de treino):")
for i, score in enumerate(mae_scores_train, 1):
    print(f"Fold {i}: {score:.4f}")

print('-' * 60)
print(f"MAE treino: {np.mean(mae_scores_train):.4f} ± {mae_train_std:.4f}")

print("\nMAE para cada fold na Cross-Validation (dados de teste):")
for i, score in enumerate(mae_scores_test, 1):
    print(f"Fold {i}: {score:.4f}")

print('-' * 60)
print(f"MAE teste: {np.mean(mae_scores_test):.4f} ± {mae_test_std:.4f}")
print('-' * 60)
print(f"MSE treino: {mse_train:.4f} ± {mse_train_std:.4f}")
print(f"MSE teste: {mse_test:.4f} ± {mse_test_std:.4f}")
print('-' * 60)
print(f"RMSE treino: {rmse_train:.4f} ± {rmse_train_std:.4f}")
print(f"RMSE teste: {rmse_test:.4f} ± {rmse_test_std:.4f}")
print('-' * 60)
print(f"R² treino: {r2_train:.4f}")
print(f"R² teste: {r2_test:.4f}")

MAE para cada fold na Cross-Validation (dados de treino):
Fold 1: 43.1828
Fold 2: 43.3422
Fold 3: 39.6237
Fold 4: 40.4944
Fold 5: 41.3972
------------------------------------------------------------
MAE treino: 41.6081 ± 1.4635

MAE para cada fold na Cross-Validation (dados de teste):
Fold 1: 42.0543
Fold 2: 45.2809
Fold 3: 45.2541
Fold 4: 43.4433
Fold 5: 41.8344
------------------------------------------------------------
MAE teste: 43.5734 ± 1.4892
------------------------------------------------------------
MSE treino: 4033.1907 ± 240.0610
MSE teste: 3763.8852 ± 500.0358
------------------------------------------------------------
RMSE treino: 63.5074 ± 1.9154
RMSE teste: 61.3505 ± 4.0013
------------------------------------------------------------
R² treino: 0.9864
R² teste: 0.9870
